<a href="https://colab.research.google.com/github/ridoy1211/Flyrank-Internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ridoy1211/Flyrank-Internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# ============================================
# SETUP
# ============================================

%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Get Hugging Face token from Colab Secret
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add your Hugging Face READ token "
        "to Colab Secrets with the name HF_TOKEN."
    )

# DuckDB connection
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print("Setup complete.")
print("HF token loaded:", bool(HF_TOKEN))

Setup complete.
HF token loaded: True


In [3]:
# ============================================
# Basic warehouse check
# ============================================

print(
    con.sql(f"""
        SELECT
            COUNT(*) AS n,
            MIN(report_date) AS min_date,
            MAX(report_date) AS max_date
        FROM {TABLES["fact_daily"]}
        WHERE month = '2026-03'
    """).df()
)

print("\nMarch 2026 rows checked.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

         n   min_date   max_date
0  9841378 2026-03-01 2026-03-31

March 2026 rows checked.


## 1. My rule and its reason codes

### Signal checks before the rule

Continuing the Great Decoupling framing from Weeks 1–3, I'll test two signals my baseline rule leans on, each with a bucket table and `n`:

1. **CTR vs. average position** — linked to FlyRank's CTR-fix flag logic. If CTR doesn't actually vary with position, comparing a page's CTR to a flat benchmark (instead of a position-adjusted one) would be the wrong design for the rule.
2. **Search volume (visibility)** — linked to the quick-win / volume flag logic. This tests whether raw traffic volume predicts engagement strongly enough to earn a graded role in the score, rather than just a simple floor.

### Proposed baseline rule

The rule scores each page's **decoupling risk using only prior-window (February 2026) data** — never the March outcome — so it's evaluated the same honest way a real predictive model would be in Week 5: a page is prioritized if it already has real visibility and its click-through rate is already below what's typical for pages at its position, in the month *before* the comparison window used to check for decoupling. Every scored page gets one `score`, one `reason_code`, and one `action` label. No fitted weights, no future-window inputs.

In [ ]:
# Signal checks and rule reasoning are stated above.
pass


In [4]:
# ============================================
# Inspect fields used by the baseline
# ============================================

print(
    con.sql(f"""
        DESCRIBE
        SELECT *
        FROM {TABLES["dim_content"]}
    """).df()[["column_name", "column_type"]]
)

print("\nDaily performance fields:")

print(
    con.sql(f"""
        DESCRIBE
        SELECT *
        FROM {TABLES["fact_daily"]}
    """).df()[["column_name", "column_type"]]
)


                   column_name column_type
0               client_hash_id     VARCHAR
1              content_hash_id     VARCHAR
2              keyword_hash_id     VARCHAR
3                  url_hash_id     VARCHAR
4           keyword_char_count      BIGINT
5          keyword_token_count      BIGINT
6               url_char_count      BIGINT
7         content_created_date        DATE
8         content_updated_date        DATE
9                 content_type     VARCHAR
10               search_volume      BIGINT
11                 competition      DOUBLE
12           competition_level     VARCHAR
13                         cpc      DOUBLE
14                 main_intent     VARCHAR
15                   backlinks      BIGINT
16              category_count      BIGINT
17        keyword_created_date        DATE
18               provider_used     VARCHAR
19                  model_used     VARCHAR
20                  char_count      BIGINT
21                  word_count      BIGINT
22         

### Signal 1 — CTR vs. average position

**FlyRank flag connection:** CTR-fix logic.

**Question:** Do pages with stronger average search positions show meaningfully different CTR from pages with weaker positions? I'm checking this on the March 2026 partition (a single mid-panel month, cheap to query) — this validates a general mechanism, not something specific to the month my rule will actually score from.

This is a signal audit only. The outcome informs how the rule is designed; it is not used as a scoring input itself.

In [7]:
# ============================================
# Signal 1 — March page-level GSC metrics
# ============================================

page_march = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(f.gsc_impressions) AS impressions,
        SUM(f.gsc_clicks) AS clicks,

        CASE
            WHEN SUM(f.gsc_impressions) > 0
            THEN SUM(f.gsc_sum_position) / SUM(f.gsc_impressions)
            ELSE NULL
        END AS avg_position

    FROM {TABLES["fact_daily"]} f

    WHERE f.month = '2026-03'
      AND f.gsc_data_available = TRUE

    GROUP BY
        f.client_hash_id,
        f.content_hash_id

    HAVING SUM(f.gsc_impressions) > 0
""").df()

page_march["ctr"] = (
    page_march["clicks"] /
    page_march["impressions"] * 100
)

print(f"Page-level rows: {len(page_march):,}")

display(page_march.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level rows: 176,738


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,6.893301,0.107313
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,3.214128,0.000000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.535346,0.106572
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.435680,0.262945
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,3.871795,0.233100


In [6]:
# ============================================
# Signal 1 — Position buckets
# ============================================

page_march["position_bucket"] = pd.cut(
    page_march["avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"],
    include_lowest=True
)

position_signal = (
    page_march
    .groupby("position_bucket", observed=False)
    .agg(
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        n=("content_hash_id", "count")
    )
    .reset_index()
)

display(position_signal)

,position_bucket,mean_ctr,median_ctr,impressions,clicks,n
0,1-3,1.169594,0.0,41420140.0,160562.0,18860
1,4-10,0.487282,0.0,148112436.0,481189.0,83288
2,11-20,0.328465,0.0,31191659.0,98488.0,29922
3,21-50,0.237885,0.0,57614383.0,80635.0,32240
4,51+,0.084632,0.0,2318971.0,958.0,12428


In [8]:
# ============================================
# Signal 1 — Honest verdict helper
# ============================================

ordered_ctr = (
    position_signal
    .dropna(subset=["mean_ctr"])
    ["mean_ctr"]
    .to_numpy()
)

if len(ordered_ctr) >= 3:
    decreases = np.sum(np.diff(ordered_ctr) < 0)
    increases = np.sum(np.diff(ordered_ctr) > 0)

    if decreases >= len(ordered_ctr) - 1:
        signal1_verdict = "CONFIRMED"
    elif increases >= len(ordered_ctr) - 1:
        signal1_verdict = "OPPOSITE"
    else:
        signal1_verdict = "MIXED"
else:
    signal1_verdict = "FALSE"

print("Signal 1 verdict:", signal1_verdict)

Signal 1 verdict: CONFIRMED


### Signal 1 verdict — CONFIRMED

Mean CTR drops monotonically from 1.17% (position 1-3, n=18,860) down to 0.08% (position 51+, n=12,428) — a clean, large-sample confirmation. **Design decision:** the rule will compare each page's CTR against the median CTR *for its own position bucket*, not against one flat benchmark — comparing a position-51 page to a position-3 page's expected CTR would be a meaningless comparison, and this check is why.

### Signal 2 — Search volume (visibility)

**FlyRank flag connection:** quick-win / volume logic.

**Question:** Does higher search volume correspond to stronger engagement (impressions/CTR) in this March slice? `search_volume` comes from `dim_content` and is a static content property, not a performance outcome.

In [14]:
# ============================================
# Signal 2 — Add search volume
# ============================================

volume_data = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        search_volume,
        content_type,
        main_intent
    FROM {TABLES["dim_content"]}
""").df()

# Ensure 'search_volume' exists in volume_data after SQL query
# This check is defensive against unexpected data retrieval issues.
if 'search_volume' not in volume_data.columns:
    print("Warning: 'search_volume' column missing from volume_data. Adding as NaN.")
    volume_data['search_volume'] = np.nan # Add with NaNs to prevent KeyError downstream

page_march = page_march.merge(
    volume_data,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# After merge, ensure the column is there for display, even if all NaNs.
# This handles cases where the merge itself might unexpectedly drop columns (unlikely for left merge)
# or if 'search_volume' was only in some rows of volume_data, leading to unexpected behavior.
if 'search_volume' not in page_march.columns:
    print("Warning: 'search_volume' column still missing from page_march after merge. Adding as NaN.")
    page_march['search_volume'] = np.nan

print("Rows after content join:", len(page_march))
display(
    page_march[
        [
            "content_hash_id",
            "impressions",
            "clicks",
            "ctr",
            "avg_position",
            "search_volume"
        ]
    ].head()
)

Rows after content join: 176738


,content_hash_id,impressions,clicks,ctr,avg_position,search_volume
0,content_7a105f548d9c6916,6523.0,7.0,0.107313,6.893301,20
1,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.214128,40
2,content_36c36abc7650d7af,5630.0,6.0,0.106572,6.535346,10
3,content_a7da352b73b02668,4944.0,13.0,0.262945,7.435680,10
4,content_1855a661b4d36130,429.0,1.0,0.233100,3.871795,50


In [ ]:
# ============================================
# Signal 2 — Search-volume buckets
# ============================================

page_march["volume_bucket"] = pd.qcut(
    page_march["search_volume"],
    q=4,
    duplicates="drop"
)
n_bins = page_march["volume_bucket"].cat.categories.size
bucket_labels = {1: ["All"], 2: ["Low", "High"], 3: ["Low", "Mid", "High"],
                 4: ["Q1 Low", "Q2", "Q3", "Q4 High"]}.get(n_bins, [f"Q{i+1}" for i in range(n_bins)])
page_march["volume_bucket"] = page_march["volume_bucket"].cat.rename_categories(bucket_labels)

volume_signal = (
    page_march
    .groupby("volume_bucket", observed=False)
    .agg(
        mean_impressions=("impressions", "mean"),
        median_impressions=("impressions", "median"),
        mean_ctr=("ctr", "mean"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        n=("content_hash_id", "count")
    )
    .reset_index()
)

display(volume_signal)


In [21]:
# ============================================
# Signal 2 — Honest verdict
# ============================================

ordered_volume = (
    volume_signal
    .dropna(subset=["mean_impressions"])
    ["mean_impressions"]
    .to_numpy()
)

if len(ordered_volume) >= 3:
    increases = np.sum(np.diff(ordered_volume) > 0)
    decreases = np.sum(np.diff(ordered_volume) < 0)

    if increases >= len(ordered_volume) - 1:
        signal2_verdict = "CONFIRMED"
    elif decreases >= len(ordered_volume) - 1:
        signal2_verdict = "OPPOSITE"
    else:
        signal2_verdict = "MIXED"
else:
    signal2_verdict = "FALSE"

print("Signal 2 verdict:", signal2_verdict)

Signal 2 verdict: MIXED


### Signal 2 verdict — MIXED

Mean impressions do **not** increase monotonically across volume quartiles (1,694 → 1,870 → 1,768 across Low/Mid/High, n = 106,221 / 21,445 / 33,140) — this echoes the exact finding from Week 1's starter-CSV analysis (affected pages had almost the same median impressions as unaffected ones). **Design decision:** because this is genuinely mixed, `search_volume` will **not** get a graded role in the score. This is a deliberate exclusion, not a failed check — a clearly-explained negative here is a real result, and it keeps the rule from prioritizing pages based on a signal that doesn't behave predictably in this data.

## 2. Build the ranked queue

### Baseline rule (prior-window only)

Using **only February 2026** — the same honest, prior-window discipline established in Weeks 2–3 (`impressions_prior30`, `clicks_prior30`, `avg_position_prior30`):

```
visible          = impressions_prior30 >= median(impressions_prior30)
ctr_gap_prior30  = position_bucket_median_ctr_prior30 - ctr_prior30   (clipped at 0)
score            = visible * ctr_gap_prior30
```

A page scores above zero only if it already has real visibility *and* its February CTR already sits below what's typical for pages at its own position — exactly the leading pattern behind Great Decoupling, built entirely from data that would have been knowable before March even started.

In [ ]:
FEB = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

feb_page = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impressions_prior30,
        SUM(gsc_clicks)      AS clicks_prior30,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_sum_position) / SUM(gsc_impressions)
             ELSE NULL END AS avg_position_prior30
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) > 0
""").df()

feb_page["ctr_prior30"] = feb_page["clicks_prior30"] / feb_page["impressions_prior30"] * 100

print(f'February (prior-window) page-level rows: {len(feb_page):,}')
feb_page.head()


In [ ]:
# Position buckets and the position-relative CTR gap, computed ENTIRELY from February data
feb_page["position_bucket"] = pd.cut(
    feb_page["avg_position_prior30"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"],
    include_lowest=True
)

position_ctr_ref = (
    feb_page.groupby("position_bucket", observed=False)["ctr_prior30"]
    .median().rename("position_median_ctr_prior30").reset_index()
)
feb_page = feb_page.merge(position_ctr_ref, on="position_bucket", how="left")

visibility_threshold = feb_page["impressions_prior30"].median()
feb_page["visible"] = (feb_page["impressions_prior30"] >= visibility_threshold).astype(int)
feb_page["ctr_gap_prior30"] = (
    feb_page["position_median_ctr_prior30"] - feb_page["ctr_prior30"]
).clip(lower=0)

feb_page["score"] = feb_page["visible"] * feb_page["ctr_gap_prior30"]

feb_page["reason_code"] = np.where(
    feb_page["score"] > 0, "visible_below_position_ctr_expectation_prior_window", "not_prioritized"
)
feb_page["action"] = np.where(
    feb_page["score"] > 0, "REVIEW_FOR_DECOUPLING_RISK", "NO_ACTION"
)

feb_page = feb_page.sort_values(["score", "impressions_prior30"], ascending=[False, False]).reset_index(drop=True)
feb_page["rank"] = np.arange(1, len(feb_page) + 1)

print(f'Visibility threshold (median Feb impressions): {visibility_threshold:,.1f}')
print(f'Prioritized pages (score > 0): {(feb_page["score"] > 0).sum():,} of {len(feb_page):,}')
feb_page[["rank","client_hash_id","content_hash_id","impressions_prior30","ctr_prior30",
          "position_median_ctr_prior30","ctr_gap_prior30","score","reason_code","action"]].head(10)


### Evaluating the rule — precision@K against the real label

The rule above never touches March data. Now, **only for evaluation**, I build the same `decoupling_signature` label from Weeks 2–3 (Feb vs. March comparison) and check how well the February-only rule ranks toward it — this is the number a Week 5 model has to beat.

In [ ]:
mar_page = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_last30,
           SUM(gsc_clicks)      AS clicks_last30
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
""").df()

eval_df = feb_page.merge(mar_page, on=["client_hash_id", "content_hash_id"], how="inner")

eval_df["impr_change_pct"] = 100 * (eval_df["impressions_last30"] - eval_df["impressions_prior30"]) / eval_df["impressions_prior30"]
eval_df["click_change_pct"] = 100 * (eval_df["clicks_last30"] - eval_df["clicks_prior30"]) / eval_df["clicks_prior30"].replace(0, np.nan)
eval_df["decoupling_signature"] = (
    eval_df["impr_change_pct"].between(-10, 10) & (eval_df["click_change_pct"] <= -15)
).astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = eval_df["decoupling_signature"].mean()
p50 = precision_at_k(eval_df["score"].values, eval_df["decoupling_signature"].values, 50)

print(f'Base rate (random-pick precision): {base_rate:.4f}')
print(f'Rule precision@50:                 {p50:.4f}')
print(f'Lift over base rate:               {p50 / base_rate:.2f}x')


In [ ]:
# ============================================
# Write ranked queue (Feb-only inputs; label/eval columns kept separate on purpose)
# ============================================
import os
os.makedirs("work/outputs", exist_ok=True)

output_columns = [
    "rank", "client_hash_id", "content_hash_id",
    "impressions_prior30", "clicks_prior30", "ctr_prior30", "avg_position_prior30",
    "score", "reason_code", "action",
]
queue = feb_page[output_columns].copy()
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Wrote: work/outputs/baseline_action_score.csv")
print("Rows:", len(queue))


## 3. Top-10 review

For each of the top 10 pages by score, reviewed with a skeptic's eye — including whether it actually turned out to match `decoupling_signature` in March, shown honestly either way.

In [ ]:
top10 = eval_df.sort_values("score", ascending=False).head(10).reset_index(drop=True)
top10["rank"] = np.arange(1, len(top10) + 1)

for _, row in top10.iterrows():
    hit = "MATCHED decoupling_signature" if row["decoupling_signature"] == 1 else "did NOT match (false positive)"
    print(f"{int(row['rank'])}. Action={row['action']} | Reason={row['reason_code']} | Score={row['score']:.3f}")
    print(f"   Why: Feb CTR {row['ctr_prior30']:.3f}% vs. its position bucket's median "
          f"{row['position_median_ctr_prior30']:.3f}%, at position {row['avg_position_prior30']:.1f}, "
          f"with {row['impressions_prior30']:,.0f} Feb impressions (above the visibility floor).")
    print(f"   Outcome check: {hit}.")
    print("   What would make it wrong: the Feb CTR gap could reflect query mix, brand-vs-non-brand "
          "mix, a SERP feature change, or simple month-to-month noise rather than a real, fixable "
          "content problem — and even a real gap doesn't prove editing the page would close it.")
    print()


## 4. Weak picks + leakage check

### Weak-pick review

A useful baseline should surface some questionable picks, not just clean wins.

In [ ]:
false_positives = top10[top10["decoupling_signature"] == 0]
print(f'{len(false_positives)} of the top 10 did not match decoupling_signature in March:\n')
for _, row in false_positives.iterrows():
    print(f"Rank {int(row['rank'])}: score={row['score']:.3f}, position={row['avg_position_prior30']:.1f}, "
          f"Feb CTR={row['ctr_prior30']:.3f}%")
    print("  Why it may be weak: a real Feb CTR gap doesn't guarantee the page keeps declining into "
          "March — it may have already recovered, or the gap may reflect something the rule can't see.")
    print()
if len(false_positives) == 0:
    print("All top 10 matched the label this run — worth rechecking with a lower score threshold "
          "or a larger K, since a rule that never misses at the very top is a reason to look harder, "
          "not a reason to relax.")


In [ ]:
# ============================================
# Real leakage audit — checked against the actual columns used, not a hardcoded string list
# ============================================
rule_input_columns = [
    "client_hash_id", "content_hash_id", "impressions_prior30", "clicks_prior30",
    "avg_position_prior30", "ctr_prior30", "position_bucket", "position_median_ctr_prior30",
    "visible", "ctr_gap_prior30", "score", "reason_code", "action", "rank",
]
forbidden = ["impressions_last30", "clicks_last30", "decoupling_signature", "trend_direction", "trend_pct"]

leaked = [c for c in forbidden if c in rule_input_columns]
print("Columns actually used to compute the score:", rule_input_columns)
print("Forbidden (label-derived / future-window) columns found in that list:", leaked or "none")
assert not leaked, "Leakage detected: a forbidden column made it into the scoring inputs."

print("\nPartitions touched for scoring (FEB only):", FEB)
print("Partition touched only for the separate evaluation label (MAR):", MAR)
print("fact_content_query_90d used anywhere:", "no")
print("Final sealed month (_sample / June 2026) used anywhere:", "no")


## Baseline summary

This notebook produced a transparent, hand-written baseline — no fitted weights — built entirely from February 2026 (prior-window) data, and evaluated against the March-based `decoupling_signature` label established in Weeks 2–3.

### Signals
- **CTR vs. position: CONFIRMED** (large-sample, monotonic) — used to build the position-relative `ctr_gap_prior30` term.
- **Search volume: MIXED** — deliberately excluded from the score; kept only as a simple visibility floor via `impressions_prior30`.

### Rule
`score = visible × ctr_gap_prior30`, using only data knowable before March.

### Evaluation
Precision@50 vs. base rate printed above — the number a Week 5 model must beat, on the exact same features, label, and slice.

### Output
`work/outputs/baseline_action_score.csv` — regenerated by this notebook on every run, intentionally not committed to git.

### Limitation
This is a decision-support ranking, not proof that fixing a flagged page's CTR gap would prevent or reverse decoupling — the top-10 review above records what would make each pick wrong.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.